## Load Dataset

In [1]:
# Load semua CSV dari Google Drive
from pathlib import Path
import pandas as pd

# 1) Mount Google Drive
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive')
except Exception:

    DRIVE_ROOT = Path('/content/drive/MyDrive')
    print('google.colab tidak terdeteksi. Pastikan path DRIVE_ROOT sesuai environment Anda.')

# 2) Path folder dataset
dataset_dir = DRIVE_ROOT / 'Purwadhika' / 'Final Project' / 'Dataset' / 'Olist Ecommerce'

# 3) Ambil semua file CSV (termasuk subfolder)
csv_files = sorted(dataset_dir.rglob('*.csv'))

if not csv_files:
    raise FileNotFoundError(
        f'Tidak ada file CSV ditemukan di: {dataset_dir}\n'
        'Cek lagi penulisan folder (huruf besar-kecil/spasi), atau pastikan Drive sudah termount.'
    )

# 4) Load semua CSV ke dictionary: key = nama file tanpa .csv
dfs = {}
for f in csv_files:
    key = f.stem
    # Jika ada nama file sama di folder berbeda, beri suffix otomatis
    original_key = key
    i = 2
    while key in dfs:
        key = f'{original_key}_{i}'
        i += 1
    dfs[key] = pd.read_csv(f)

print(f'Berhasil load {len(dfs)} file CSV dari: {dataset_dir}\n')
for name, df in dfs.items():
    print(f'- {name}: shape={df.shape}')

# Contoh akses DataFrame
# dfs['nama_file']

# Preview cepat 1 DataFrame pertama
# first_key = next(iter(dfs))
# print(f'\nPreview: {first_key}')
# display(dfs[first_key].head())

Mounted at /content/drive
Berhasil load 9 file CSV dari: /content/drive/MyDrive/Purwadhika/Final Project/Dataset/Olist Ecommerce

- olist_customers_dataset: shape=(99441, 5)
- olist_geolocation_dataset: shape=(1000163, 5)
- olist_order_items_dataset: shape=(112650, 7)
- olist_order_payments_dataset: shape=(103886, 5)
- olist_order_reviews_dataset: shape=(99224, 7)
- olist_orders_dataset: shape=(99441, 8)
- olist_products_dataset: shape=(32951, 9)
- olist_sellers_dataset: shape=(3095, 4)
- product_category_name_translation: shape=(71, 2)


## Olist Commerce Copilot

Asisten analitik e-commerce yang bisa:

1. Menjawab pertanyaan katalog/produk/review lewat RAG.
2. Menjawab pertanyaan metrik bisnis (GMV, order trend, payment type, delay shipment) lewat SQL.
3. Memberikan insight otomatis dan rekomendasi tindakan bisnis.


> GMV adalah Gross Merchandise Value yaitu total nilai seluruh transaksi penjualan yang terjadi di suatu platform sebelum dikurangi biaya apa pun.


## Arah Produk Berdasarkan 2 Persona

Project diposisikan untuk 2 tipe pengguna utama:

1. Users (end-user umum): fokus pada discovery produk, tanya jawab produk, dan rekomendasi.
2. Bisnis Mentor (pengambil keputusan): fokus pada insight bisnis, performa operasional, dan monitoring KPI.

## Data Preparation

### 1. Tujuan Data Layer

Menyediakan data konsisten untuk:

1. SQL analytics (metrik bisnis dan operasional).
2. RAG retrieval (jawaban berbasis konteks produk/review).

### 2. Source Dataset Olist yang Dipakai

1. orders
2. order_items
3. order_payments
4. order_reviews
5. customers
6. products
7. sellers
8. product_category_name_translation

In [2]:
def find_table(keyword):
    for name, df in dfs.items():
        if keyword in name.lower():
            return df.copy(), name
    raise KeyError(f'Table not found for keyword: {keyword}')

# Mapping tables
orders, orders_name = find_table('orders')
order_items, order_items_name = find_table('order_items')
payments, payments_name = find_table('order_payments')
reviews, reviews_name = find_table('order_reviews')
customers, customers_name = find_table('customers')
products, products_name = find_table('products')
sellers, sellers_name = find_table('sellers')

translation = None
translation_name = None
for name, df in dfs.items():
    if 'translation' in name.lower():
        translation = df.copy()
        translation_name = name
        break

# Mempercantik Output Mapping
table_mapping = {
    'Orders': [orders_name, orders.shape],
    'Order Items': [order_items_name, order_items.shape],
    'Payments': [payments_name, payments.shape],
    'Reviews': [reviews_name, reviews.shape],
    'Customers': [customers_name, customers.shape],
    'Products': [products_name, products.shape],
    'Sellers': [sellers_name, sellers.shape],
    'Translation': [translation_name, translation.shape if translation is not None else 'Not Found']
}

print("=" * 85)
print(f" {'VARIABLE':<15} | {'SOURCE TABLE NAME':<35} | {'SHAPE (R, C)':<20} ")
print("=" * 85)
for var, info in table_mapping.items():
    shape_str = f"{info[1][0]:,} x {info[1][1]}" if isinstance(info[1], tuple) else str(info[1])
    print(f" {var:<15} | {str(info[0]):<35} | {shape_str:<20} ")
print("=" * 85)

 VARIABLE        | SOURCE TABLE NAME                   | SHAPE (R, C)         
 Orders          | olist_orders_dataset                | 99,441 x 8           
 Order Items     | olist_order_items_dataset           | 112,650 x 7          
 Payments        | olist_order_payments_dataset        | 103,886 x 5          
 Reviews         | olist_order_reviews_dataset         | 99,224 x 7           
 Customers       | olist_customers_dataset             | 99,441 x 5           
 Products        | olist_products_dataset              | 32,951 x 9           
 Sellers         | olist_sellers_dataset               | 3,095 x 4            
 Translation     | product_category_name_translation   | 71 x 2               


## 3. Generate Product Name via LLM

Metode ini menggunakan OpenAI GPT untuk menghasilkan nama produk yang lebih komersial dan natural

**Catatan:** Pastikan Anda telah menambahkan `OPENAI_API_KEY` di bagian Secrets Colab (ikon kunci di sidebar kiri).

In [3]:
!pip install openai -q

In [4]:
import openai
from google.colab import userdata

try:
    client = openai.OpenAI(api_key=userdata.get('OPENAI_API_KEY'))
    print("OpenAI Client initialized successfully.")
except Exception as e:
    print(f"Error: {e}. Pastikan secret 'OPENAI_API_KEY' sudah diset.")

OpenAI Client initialized successfully.


In [8]:
# Try to import tqdm, and if it fails, install it
import json
try:
    from tqdm.auto import tqdm # Import tqdm
except ImportError:
    print("tqdm not found, attempting to install...")
    !pip install tqdm -q
    from tqdm.auto import tqdm
    print("tqdm installed and imported successfully.")

BATCH_SIZE = 50 # Anda bisa eksperimen dengan ukuran batch yang berbeda (misal: 20, 50, 100) untuk performa terbaik.

def get_llm_product_names_batched(batch_data):
    """Menghasilkan nama produk ID dan EN secara konsisten dalam 1 API call dan format JSON untuk batch produk."""

    # Bangun prompt untuk setiap item dalam batch
    item_prompts = []
    for i, row in enumerate(batch_data):
        category, weight, product_id = row
        item_prompts.append(
            f"{i+1}. Kategori: {category}, Berat: {weight} gram, ID Produk: {product_id[:5]}"
        )

    full_prompt = f"""Tugas: Buat nama produk e-commerce singkat (3-5 kata) dalam Bahasa Indonesia dan terjemahannya ke Bahasa Inggris secara natural untuk setiap item berikut.

Format keluaran harus berupa JSON dengan kunci utama 'products', yang berisi array objek. Setiap objek dalam array harus memiliki kunci 'name_id' untuk nama Bahasa Indonesia dan 'name_en' untuk nama Bahasa Inggris, sesuai urutan input.

Contoh keluaran:
{{
  "products": [
    {{"name_id": "Tas Selempang Kulit Pria", "name_en": "Men's Leather Sling Bag"}},
    {{"name_id": "Sepatu Lari Ringan", "name_en": "Lightweight Running Shoes"}}
  ]
}}

Items:
""" + "\n".join(item_prompts)

    json_output = "N/A" # Inisialisasi untuk penanganan error yang lebih baik
    try:
        res = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": full_prompt}],
            max_tokens=BATCH_SIZE * 150, # Perkiraan token: 100 per produk + buffer
            temperature=0.7,
            response_format={"type":"json_object"}
        )

        json_output = res.choices[0].message.content
        parsed_output = json.loads(json_output)

        results_list = parsed_output.get('products', [])

        # --- NEW LOGIC: Ensure results_list length matches batch_data length ---
        expected_len = len(batch_data)
        if len(results_list) > expected_len:
            # Truncate if LLM returned too many
            results_list = results_list[:expected_len]
        elif len(results_list) < expected_len:
            # Pad with error values if LLM returned too few
            padding_needed = expected_len - len(results_list)
            results_list.extend([{'name_id': 'Error ID', 'name_en': 'Error EN'}] * padding_needed)
        # --- END NEW LOGIC ---

        batched_names_id = [item.get('name_id', 'Error ID') for item in results_list]
        batched_names_en = [item.get('name_en', 'Error EN') for item in results_list]

        return list(zip(batched_names_id, batched_names_en))

    except Exception as e:
        print(f"Error generating batched product names: {e}. Output content: {json_output}")
        # Jika terjadi error (misal JSON tidak valid), kembalikan 'Error ID', 'Error EN' untuk setiap item dalam batch
        return [("Error ID", "Error EN")] * len(batch_data)

# ===============================================
# Proses DataFrame 'products' menggunakan Batching
# ===============================================

all_llm_product_names = []

print(f"Memulai proses LLM batched untuk {len(products)} produk unik dengan ukuran batch {BATCH_SIZE}.")

# Persiapan data untuk batching
product_data_for_batching = products[['product_category_name', 'product_weight_g', 'product_id']].values.tolist()

# Loop melalui data dalam batch
for i in tqdm(range(0, len(product_data_for_batching), BATCH_SIZE), desc="Processing batches"):
    batch = product_data_for_batching[i : i + BATCH_SIZE]
    try:
        batch_results = get_llm_product_names_batched(batch)
        all_llm_product_names.extend(batch_results)
    except KeyboardInterrupt:
        print("Proses batching dihentikan oleh pengguna.")
        break
    except Exception as e:
        print(f"Terjadi kesalahan pada batch {i}-{i+BATCH_SIZE-1}: {e}")
        # Jika terjadi error pada loop utama, tambahkan nilai error untuk setiap item di batch tersebut
        all_llm_product_names.extend([("Error ID", "Error EN")] * len(batch))

# Memastikan panjang hasil sesuai dengan jumlah produk yang diproses (termasuk interupsi)
final_results_len = len(all_llm_product_names)

# Jika hasil kurang dari total produk, isi sisanya dengan nilai kosong
if final_results_len < len(products):
    print(f"Peringatan: Hanya {final_results_len} dari {len(products)} produk yang berhasil diproses. Mengisi sisanya dengan nilai kosong.")
    remaining_count = len(products) - final_results_len
    all_llm_product_names.extend([('','')]* remaining_count) # Corrected line

# Memisahkan hasil menjadi dua kolom baru
products['llm_product_name_id'] = [res[0] for res in all_llm_product_names]
products['llm_product_name_en'] = [res[1] for res in all_llm_product_names]

print("Proses LLM Batched Selesai.")
display(products[['product_id', 'llm_product_name_id', 'llm_product_name_en']].head())

Memulai proses LLM batched untuk 32951 produk unik dengan ukuran batch 50.


Processing batches:   0%|          | 0/660 [00:00<?, ?it/s]

Proses LLM Batched Selesai.


,product_id,llm_product_name_id,llm_product_name_en
0,1e9e8ef04dbcff4541ed26657ea517e5,Parfum Aroma Segar,Fresh Aroma Perfume
1,3aa071139cb16b67ca9e5dea641aaa2f,Kerajinan Tangan Unik,Unique Handcrafted Art
2,96bd76ec8810374ed1b65e291975717f,Aksesori Olahraga,Sports Accessories
3,cef67bcfe19066a932b7673e239eb23d,Perlengkapan Bayi,Baby Supplies
4,9dc1a7de274444849c219cff195d0b71,Alat Rumah Tangga,Household Utilities


In [12]:
# Path untuk menyimpan CSV hasil LLM
# output_csv_path = DRIVE_ROOT / 'Purwadhika' / 'Final Project' / 'Dataset' / 'Olist Ecommerce' / 'products_with_llm_names.csv'
output_csv_path = DRIVE_ROOT / 'Purwadhika' / 'Final Project' / 'Sprint 1 - Data Foundation' / '4. Init Projects' / 'products_with_llm_names.csv'

# Simpan DataFrame 'products' ke CSV
products.to_csv(output_csv_path, index=False)

print(f"DataFrame 'products' dengan nama LLM telah disimpan ke: {output_csv_path}")

DataFrame 'products' dengan nama LLM telah disimpan ke: /content/drive/MyDrive/Purwadhika/Final Project/Sprint 1 - Data Foundation/4. Init Projects/products_with_llm_names.csv


## 4. Data Processing for Aiven MySQL
Target output pada section ini adalah `fact_olist_order_line` yang siap disimpan ke Aiven MySQL.

Aktivitas utama:
1. Join tabel inti transaksi.
2. Bentuk kolom numerik dan timestamp yang konsisten.
3. Hitung field turunan seperti `total_order_value`, `delay_days`, dan `is_late`.

Column :

1. order_id
2. order_item_id
3. product_id
4. seller_id
5. shipping_limit_date
6. item_price
7. freight_value
8. customer_id
9. order_status
10. purchase_ts
11. approved_ts
12. order_delivered_carrier_date
13. delivered_customer_ts
14. estimated_delivery_ts
15. payment_value
16. payment_type
17. review_score
18. review_comment_title
19. review_comment_message
20. customer_unique_id
21. customer_city
22. customer_state
23. product_category_name
24. product_category_name_english
25. llm_product_name_id,
26. llm_product_name_en,
27. seller_city
28. seller_state
29. total_order_value
30. delay_days
31. is_late




In [9]:
import pandas as pd
import numpy as np

# 1. Agregasi Payments
payments_agg = payments.groupby('order_id').agg({
    'payment_value': 'sum',
    'payment_type': lambda x: '/'.join(x.unique())
}).reset_index().rename(columns={'payment_value': 'total_order_value'})

# 2. Review terbaru
reviews_latest = reviews.sort_values('review_answer_timestamp', ascending=False).drop_duplicates('order_id')

# 3. Join Utama
fact_table = pd.merge(order_items, orders, on='order_id', how='left')
fact_table = pd.merge(fact_table, payments_agg, on='order_id', how='left')
fact_table = pd.merge(fact_table, customers, on='customer_id', how='left')
fact_table = pd.merge(fact_table, sellers, on='seller_id', how='left')

# Gabungkan dengan products (yang sekarang sudah punya kolom LLM) dan translation
# Pastikan products sudah memiliki llm_product_name_id dan llm_product_name_en
prod_full = pd.merge(products, translation, on='product_category_name', how='left')
fact_table = pd.merge(fact_table, prod_full, on='product_id', how='left')
fact_table = pd.merge(fact_table, reviews_latest, on='order_id', how='left')

# 4. Feature Engineering
ts_cols = ['order_purchase_timestamp', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'order_approved_at', 'shipping_limit_date']
for col in ts_cols:
    fact_table[col] = pd.to_datetime(fact_table[col])

fact_table['delay_days'] = (fact_table['order_delivered_customer_date'] - fact_table['order_estimated_delivery_date']).dt.days
fact_table['is_late'] = fact_table['delay_days'] > 0

# 5. Final Renaming & Selection sesuai spek Anda
column_mapping = {
    'order_purchase_timestamp': 'purchase_ts',
    'order_approved_at': 'approved_ts',
    'order_delivered_customer_date': 'delivered_customer_ts',
    'order_estimated_delivery_date': 'estimated_delivery_ts',
    'price': 'item_price'
}

fact_olist_order_line = fact_table.rename(columns=column_mapping)

final_cols = [
    'order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date',
    'item_price', 'freight_value', 'customer_id', 'order_status', 'purchase_ts',
    'approved_ts', 'order_delivered_carrier_date', 'delivered_customer_ts',
    'estimated_delivery_ts', 'payment_value', 'payment_type', 'review_score',
    'review_comment_title', 'review_comment_message', 'customer_unique_id',
    'customer_city', 'customer_state', 'product_category_name',
    'product_category_name_english', 'llm_product_name_id', 'llm_product_name_en',
    'seller_city', 'seller_state', 'total_order_value', 'delay_days', 'is_late'
]

# Filter columns yang benar-benar ada di DataFrame
fact_olist_order_line = fact_olist_order_line[[c for c in final_cols if c in fact_olist_order_line.columns]]

print(f"Tabel Akhir Siap: {fact_olist_order_line.shape}")
display(fact_olist_order_line.head())

Tabel Akhir Siap: (112650, 30)


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,item_price,freight_value,customer_id,order_status,purchase_ts,...,customer_state,product_category_name,product_category_name_english,llm_product_name_id,llm_product_name_en,seller_city,seller_state,total_order_value,delay_days,is_late
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,...,RJ,cool_stuff,cool_stuff,Mainan Edukatif Anak,Educational Toys for Kids,volta redonda,SP,72.19,-9.0,False
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,...,SP,pet_shop,pet_shop,Peralatan Dapur Modern,Modern Kitchen Equipment,sao paulo,SP,259.83,-3.0,False
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,...,MG,moveis_decoracao,furniture_decor,Dekorasi Furnitur Mewah,Luxury Furniture Decoration,borda da mata,MG,216.87,-14.0,False
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,...,SP,perfumaria,perfumery,Parfum Elegan Wanita,Elegant Women's Perfume,franca,SP,25.78,-6.0,False
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,...,SP,ferramentas_jardim,garden_tools,Peralatan Kebun Berkualitas,Quality Garden Tools,loanda,PR,218.04,-16.0,False


In [13]:
# Simpan fact_olist_order_line ke csv
# Path untuk menyimpan CSV hasil LLM
output_csv_path = DRIVE_ROOT / 'Purwadhika' / 'Final Project' / 'Sprint 1 - Data Foundation' / '4. Init Projects' / 'fact_olist_order_line.csv'

# Simpan DataFrame 'products' ke CSV
fact_olist_order_line.to_csv(output_csv_path, index=False)

print(f"DataFrame 'fact_olist_order_line' dengan nama LLM telah disimpan ke: {output_csv_path}")

DataFrame 'fact_olist_order_line' dengan nama LLM telah disimpan ke: /content/drive/MyDrive/Purwadhika/Final Project/Sprint 1 - Data Foundation/4. Init Projects/fact_olist_order_line.csv


<!--

### 3. SQL Analytical Table
Nama tabel: fact_olist_order_line

Platform : Aiven MysQl

Kolom minimal:

1. order_id
2. order_item_id
3. product_id
4. seller_id
5. customer_id
6. customer_unique_id
7. purchase_ts
8. approved_ts
9. delivered_customer_ts
10. estimated_delivery_ts
11. order_status
12. payment_type
13. payment_value
14. item_price
15. freight_value
16. total_order_value
17. product_category_name
18. product_category_english
19. seller_city
20. seller_state
21. customer_city
22. customer_state
23. review_score
24. review_comment_title
25. review_comment_message
26. delay_days
27. is_late -->